# Ratio-CATE Learner Benchmark

Evaluation of meta-learners for ratio-based CATE estimation:
$$\tau(x) = \frac{E[Y|W=1,X]}{E[Y|W=0,X]}$$

## Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)

%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datasets import ALL_DATASETS, RCT_DATASETS, OBS_DATASETS, CONVERSION_RATES
from learner import ALL_LEARNER
from benchmark import run_benchmark
from visualization import plot_heatmap, insignificant_datasets, plot_best_per_group, plot_dots_with_lines
from tables import make_metric_table


## Configuration

In [ ]:
# Datasets to evaluate
DATASETS = ALL_DATASETS
# Learners to evaluate
LEARNER = list(ALL_LEARNER.keys())

# Number of runs per combination
N_RUNS = 50

# Base seed (each run uses BASE_SEED + run_idx)
BASE_SEED = 42


RESULTS_CSV = 'benchmark_results.csv'

In [ ]:
print(f"Datasets: {DATASETS}")
print(f"Learners: {LEARNER}")
print(f"Runs per combination: {N_RUNS}")


## Run Benchmark

In [ ]:
df_results = run_benchmark(
    datasets=DATASETS,
    learners=LEARNER,
    n_runs=N_RUNS,
    base_seed=BASE_SEED,
    results_csv=RESULTS_CSV,
)


In [ ]:

#Especially smaller datasets suffer from larger variance across the seeds. 
#This can be seen in many gray cells in the heatmap, highlighting
#missing statistical significance. We identify such datasets and run more seeds for them

ADDITIONAL_RUNS_PER_TOPUP = 50
MAX_TOPUPS = 5            # safety cap; with 50-seed batches that's +250 max
MIN_GREY_CELLS = 3        # top up datasets with at least this many grey cells

for round_idx in range(MAX_TOPUPS):
    pending = insignificant_datasets(
        df_results, metric='qini_ratio',
        baseline='S',
        learners=LEARNER,
        datasets=DATASETS,
        min_grey_cells=MIN_GREY_CELLS,
    )
    if not pending:
        print(f"All datasets significant. Done after {round_idx} top-up round(s).")
        break

    # How many seeds does each pending dataset already have?
    have = (
        df_results[df_results['dataset'].isin(pending)]
        .groupby('dataset')['seed'].nunique()
        .to_dict()
    )
    print(f"\n=== Top-up round {round_idx + 1}: "
        f"{len(pending)} dataset(s) still have grey cells ===")
    for d in pending:
        print(f"  {d}: {have.get(d, 0)} seeds → +{ADDITIONAL_RUNS_PER_TOPUP}")

    df_results = run_benchmark(
        datasets=pending,
        learners=LEARNER,
        n_runs=max(have.values()) + ADDITIONAL_RUNS_PER_TOPUP,
        base_seed=BASE_SEED,
        results_csv='benchmark_results.csv',
    )
else:
    print(f"\nReached MAX_TOPUPS={MAX_TOPUPS}; some datasets may still be grey.")

## Export Results

In [ ]:
# Save raw results
df_results.to_csv(RESULTS_CSV, index=False)

print("Results saved")

## Visualizations

In [ ]:


LEARNER_ORDER = list(ALL_LEARNER.keys())

plt.rcParams.update({'font.size': 11, 'figure.dpi': 150})

print(f"df_results rows: {len(df_results)}")
print(f"learners present: {sorted(df_results['learner'].unique())}")
print(f"datasets present: {sorted(df_results['dataset'].unique())}")


### Heatmaps: Qini (ratio CATE)

In [ ]:
# Heatmap: Qini ratio CATE on RCT datasets
plot_heatmap(
    df_results, metric='qini_ratio',
    datasets=RCT_DATASETS,
    learner_order= LEARNER_ORDER,
    conv_rates=CONVERSION_RATES,
    savepath='fig_heatmap_qini_ratio_rct.png',
)


In [ ]:
# Heatmap: Qini ratio CATE on observational datasets
plot_heatmap(
    df_results, metric='qini_ratio',
    datasets=OBS_DATASETS,
    learner_order=LEARNER_ORDER,
    conv_rates=CONVERSION_RATES,
    savepath='fig_heatmap_qini_ratio_obs.png',
)


In [ ]:


groups = {
    'DR-S/T/Q (log)':           ['DR-S', 'DR-S log', 'DR-T', 'DR-T log', 'DR-Q', 'DR-Q log', 'DR-Q-Simple', 'DR-Q-Simple log'],
    'difference Methods (T, X, R, DR)':             ['T',  'X', 'R', 'DR']
    
}

fig, ratio = plot_dots_with_lines(
    df_results, 'qini_ratio', RCT_DATASETS,
    conv_rates=CONVERSION_RATES,
    groups=groups,
    extras=['Q', 'Q-Simple'],                          # featured learner as line
    secondary=['X', 'DR-Q-Simple'],
    regime_split_after=3,                  # split between 4th and 5th column
    savepath='fig_rct_dots_with_lines.png',
)


In [ ]:
groups = {
    'best new DR':       ['DR-S', 'DR-S log', 'DR-T', 'DR-T log',
                        'DR-Q', 'DR-Q log', 'DR-Q-Simple', 'DR-Q-Simple log'],
    'best of the rest':  ['T', 'Q', 'Q-Simple', 'X', 'DR'],   # R excluded
}

fig, best_ratio, best_winner = plot_best_per_group(
    df_results, 'qini_ratio', OBS_DATASETS,
    conv_rates=CONVERSION_RATES,
    groups=groups,
    extras=['R'],                            # R as dotted overlay
    savepath='fig_obs_new_vs_rest.png',
)

### Heatmaps: Qini (difference CATE)

In [ ]:
# Heatmap: Qini difference CATE on RCT datasets
plot_heatmap(
    df_results, metric='qini_difference',
    datasets=RCT_DATASETS,
    learner_order=LEARNER_ORDER,
    conv_rates=CONVERSION_RATES,
    savepath='fig_heatmap_qini_difference_rct.png',
)


In [ ]:
# Heatmap: Qini difference CATE on observational datasets
plot_heatmap(
    df_results, metric='qini_difference',
    datasets=OBS_DATASETS,
    learner_order=LEARNER_ORDER,
    conv_rates=CONVERSION_RATES,
    savepath='fig_heatmap_qini_difference_obs.png',
)


### Heatmaps: CalError (ratio CATE)

In [ ]:
# Heatmap: CalError ratio CATE on RCT datasets
plot_heatmap(
    df_results, metric='cal_error_ratio',
    datasets=RCT_DATASETS,
    learner_order=LEARNER_ORDER,
    conv_rates=CONVERSION_RATES,
    savepath='fig_heatmap_cal_ratio_rct.png',
)


In [ ]:
# Heatmap: CalError ratio CATE on observational datasets
plot_heatmap(
    df_results, metric='cal_error_ratio',
    datasets=OBS_DATASETS,
    learner_order=LEARNER_ORDER,
    conv_rates=CONVERSION_RATES,
    savepath='fig_heatmap_cal_ratio_obs.png',
)


### Heatmaps: CalError (difference CATE)

In [ ]:
# Heatmap: CalError difference CATE on RCT datasets
plot_heatmap(
    df_results, metric='cal_error_difference',
    datasets=RCT_DATASETS,
    learner_order=LEARNER_ORDER,
    conv_rates=CONVERSION_RATES,
    savepath='fig_heatmap_cal_difference_rct.png',
)


In [ ]:
# Heatmap: CalError difference CATE on observational datasets
plot_heatmap(
    df_results, metric='cal_error_difference',
    datasets=OBS_DATASETS,
    learner_order=LEARNER_ORDER,
    conv_rates=CONVERSION_RATES,
    savepath='fig_heatmap_cal_difference_obs.png',
)


## LaTeX Tables

In [ ]:
# LaTeX tables: per-dataset Qini and CalError, ratio and difference scales

QINI_RATIO_CAPTION_TPL = (
    'Qini coefficient (ratio CATE) on {kind} datasets (mean over seeds; '
    r'SE in parentheses). Best per column in \textbf{bold}.'
    'Bottom row: mean across all learners per dataset.'
)
QINI_DIFF_CAPTION_TPL = (
    'Qini coefficient (difference CATE, cumulative-gain formulation) on {kind} datasets '
    '(mean over seeds; SE in parentheses). '
    r'Best per column in \textbf{bold}.'
)
CAL_RATIO_CAPTION_TPL = (
    'Multiplicative calibration error (ratio CATE) on {kind} datasets '
    '(mean over seeds; SE in parentheses). '
    r'Best per column in \textbf{bold}.'
)
CAL_DIFF_CAPTION_TPL = (
    'Calibration error (difference CATE) on {kind} datasets '
    '(mean over seeds; SE in parentheses). '
    r'Best per column in \textbf{bold}.'
)


def emit(metric_key, datasets, kind, label, caption_tpl):
    print(make_metric_table(
        df_results, metric=metric_key, datasets=datasets,
        caption=caption_tpl.replace('{kind}', kind), label=label,
    ))
    print('\n\n')


# Qini ratio
emit('qini_ratio', RCT_DATASETS,      'RCT',           'tab:qini-ratio-rct', QINI_RATIO_CAPTION_TPL)
emit('qini_ratio', OBS_DATASETS, 'observational', 'tab:qini-ratio-obs', QINI_RATIO_CAPTION_TPL)

# Qini difference
#emit('qini_difference', RCT_DATASETS,      'RCT',           'tab:qini-diff-rct', QINI_DIFF_CAPTION_TPL)
#emit('qini_difference', OBS_DATASETS, 'observational', 'tab:qini-diff-obs', QINI_DIFF_CAPTION_TPL)

# CalError ratio
emit('cal_error_ratio', RCT_DATASETS,      'RCT',           'tab:cal-ratio-rct', CAL_RATIO_CAPTION_TPL)
emit('cal_error_ratio', OBS_DATASETS, 'observational', 'tab:cal-ratio-obs', CAL_RATIO_CAPTION_TPL)

# CalError difference
#emit('cal_error_difference', RCT_DATASETS,      'RCT',           'tab:cal-diff-rct', CAL_DIFF_CAPTION_TPL)
#emit('cal_error_difference', OBS_DATASETS, 'observational', 'tab:cal-diff-obs', CAL_DIFF_CAPTION_TPL)
